# Tutorial: BESS shortlist workflow

Audience:
- Analysts who need a repeatable path from GIS screening to BESS revenue review.

Prerequisites:
- `luminus-py` installed with notebook dependencies.
- A working `luminus-mcp` install on `PATH`.
- Access to both the GIS and BESS profiles used by your environment.

Learning goals:
- Reuse GIS ranking output to shape a shortlist.
- Carry the shortlist into BESS revenue estimation.
- Keep the analyst handoff compact and skimmable.
- Produce a notebook a portfolio team can read without extra context.


## Outline

1. Start GIS and BESS profile clients.
2. Define a compact candidate set.
3. Rank the candidates with the GIS profile.
4. Carry the ranked shortlist into BESS revenue calls.
5. Capture the result for review.


In [ ]:
from __future__ import annotations

from luminus import Luminus
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

gis = Luminus(profile="gis", request_timeout=120.0)
bess = Luminus(profile="bess")
gis, bess


## Step 1 - Define the candidate list

The shortlist starts as a small, human-readable table. That makes it easy to track what was screened and why it survived to the next stage.


In [ ]:
candidate_sites = pd.DataFrame(
    [
        {"label": "A", "site_name": "Alpha", "lat": 52.520, "lon": 13.405},
        {"label": "B", "site_name": "Bravo", "lat": 52.470, "lon": 13.315},
        {"label": "C", "site_name": "Charlie", "lat": 52.610, "lon": 13.220},
        {"label": "D", "site_name": "Delta", "lat": 52.680, "lon": 13.480},
        {"label": "E", "site_name": "Echo", "lat": 52.430, "lon": 13.550},
    ]
)

display(candidate_sites)
candidate_sites.to_dict(orient="records")


## Step 2 - Rank the sites with the GIS profile

The ranking table tells you which sites deserve further diligence. The notebook keeps the full frame visible so reviewers can see the exact shortlist that fed the revenue step.


In [ ]:
comparison = gis.compare_sites(country="DE", sites=candidate_sites.to_dict(orient="records"))
rankings = comparison.to_pandas(data_key="rankings")

display(rankings.head())
rankings.columns.tolist()


## Step 3 - Carry the ranking into BESS review

Carry the ranked `label` values forward so the shortlist really reflects the GIS ordering. Keep a separate `site_name` column for human-readable output.


In [ ]:
if "label" not in rankings.columns:
    raise KeyError("Expected compare_sites rankings to include label values.")

ordered_labels = rankings["label"].head(3).tolist()
shortlist = candidate_sites.set_index("label").reindex(ordered_labels).reset_index()

display(shortlist)


## Step 4 - Estimate revenue for the shortlist

The BESS profile turns the shortlist into a revenue review. Use a market zone that matches the shortlist geography, then collect each response into a tidy frame for side-by-side comparison.


In [ ]:
revenue_frames: list[pd.DataFrame] = []

for site in shortlist.itertuples(index=False):
    estimate = bess.estimate_site_revenue(lat=site.lat, lon=site.lon, zone="DE", technology="bess")
    frame = estimate.to_pandas()
    frame["site"] = site.site_name
    frame["site_label"] = site.label
    frame["lat"] = site.lat
    frame["lon"] = site.lon
    revenue_frames.append(frame)

revenue_df = pd.concat(revenue_frames, ignore_index=True)
display(revenue_df.head())
revenue_df.columns.tolist()


## Exercises

- Add another candidate and see whether it survives into the BESS shortlist.
- Swap the shortlist length from three sites to five sites.
- Use the same pattern for a second country and compare the output frames.


## Pitfall

Do not wait until the BESS step to clean up the candidate list. The workflow is easier to audit when the GIS input table stays small and explicit.


## Extension

If you need a reusable screening pack, wrap the GIS ranking and BESS revenue calls in a small helper that accepts a candidate table and returns the final shortlist frame.


In [ ]:
gis.close()
bess.close()
